# GSM8K on the chat models of this computer

GSM8K asks grade-school mathematics word problems, each answered with a number ([Cobbe, Kosaraju, Bavarian, Chen, Jun, Kaiser, Plappert, Tworek, Hilton, Nakano, Hesse and Schulman (2021)](https://doi.org/10.48550/arXiv.2110.14168)). This notebook does what the page Benchmark Answering Models does (Compose > Local AI Models > Benchmark Answering Models...), with the same functions, those of [PollisBenchmarks.jl](https://github.com/Trumpingtons/PollisBenchmarks.jl): it downloads the benchmark, draws a simple random sample of its problems, has each chat model chosen answer them, alone in memory, and grades the answers. The runs are kept in `~/.pollis/benchmarks/runs`, where the page shows them too.

In [ ]:
import PollisBenchmarks as PB

## 1. The benchmark

`download` fetches GSM8K from Hugging Face into `~/.pollis/benchmarks/data` the first time: its 1,319 test problems, and the training problems with their worked answers, which a prompt can show first.

In [ ]:
info = PB.download("gsm8k")
info.count

## 2. The sample

GSM8K has no strata, so `draw` takes a simple random sample of `size` problems, drawn with `seed`: the same seed draws the same sample again.

In [ ]:
sample = PB.draw("gsm8k"; size = 20, seed = 1)
sample.size

## 3. The models

The chat models on this computer. Choose those to run by their numbers.

In [ ]:
# The chat models on this computer, in ~/.pollis/models: tick those to run by their numbers below
installed = sort([joinpath(root, file) for (root, _, files) in walkdir(joinpath(homedir(), ".pollis", "models"))
	for file in files if endswith(file, ".gguf") && !occursin(r"mmproj|embed|rerank"i, file)])
for (number, file) in enumerate(installed)
	println(number, ". ", basename(file))
end

In [ ]:
models = installed[[1]]

## 4. The run

`new_run` keeps the run with its settings: the worked training problems shown before each problem (`shots`), the answer worked step by step or alone (`reasoning`), thinking for the models that think, and the most tokens of an answer. The answer read is the number of "The answer is N", or else the last number of the answer. `run_models!` then starts each model's llama-server, has it answer the problems, each graded as it comes, and stops it before the next.

In [ ]:
run = PB.new_run("gsm8k", sample; shots = 0, reasoning = true, thinking = false, max_tokens = 2048)
figures = PB.run_models!(run, models; progress = (model, done, total, tokens) -> print("\r", basename(model), ": ", done, " of ", total, "   "))

## 5. The results

In [ ]:
# Each model's score with its 95% interval, the answers not read or cut off, its speed and its memory
percent(value) = value === nothing ? "-" : string(round(100 * value; digits = 1), "%")
for model in figures
	println(model.label)
	println("  Score ", percent(model.score), model.half === nothing ? "" : " ± " * string(round(100 * model.half; digits = 1)),
		"  (", model.right, " of ", model.questions, " right; ", model.unread, " not read, ", model.cutOff, " cut off)")
	println("  First token ", round(Int, something(model.firstTokenMs, 0)), " ms, writing ",
		round(something(model.writingSpeed, 0); digits = 1), " tokens/s, ", round(something(model.seconds, 0); digits = 1), " s an answer")
	model.memory === nothing || println("  Memory ", round(model.memory / 1024^3; digits = 1), " GB")
end

In [ ]:
# The first answers, with the number read from each and whether it is right
for answer in first(PB.load_answers(run), 3)
	println(answer.id, ": read ", something(get(answer, :extracted, nothing), "nothing"), get(answer, :correct, false) ? ", right" : ", wrong")
	println(first(get(answer, :text, get(answer, :error, "")), 600), "\n")
end